# LeetCode 795 — Number of Subarrays with Bounded Maximum bằng Python

Link đề: https://leetcode.com/problems/number-of-subarrays-with-bounded-maximum/ · Độ khó: **Medium** · Chủ đề: Array, Two Pointers

Skeleton LeetCode:

```python
class Solution:
    def numSubarrayBoundedMax(self, nums: List[int], left: int, right: int) -> int:
        ...
```

Ghi chú: các ví dụ ngoài hai ví dụ của đề (ví dụ 3 và 4 ở mục 2) do tôi tự đặt, đáp án đã kiểm chứng bằng vét cạn; nếu cần khớp từng chữ, hãy đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm số **mảng con liên tiếp, không rỗng** của `nums` mà **phần tử lớn nhất** trong mảng con đó nằm trong đoạn `[left, right]`.
2. Nắm các công cụ Python: duyệt mọi mảng con bằng hai vòng `for`, duy trì **max chạy** (`running max`), công thức đếm mảng con của một đoạn liên tiếp `L(L + 1) // 2`, **hai con trỏ/hai chỉ số "gần nhất"**, **ngăn xếp đơn điệu** (monotonic stack) và `max`/`min` trong vòng lặp.
3. Đi từ **thử mọi mảng con** → **bù trừ: đếm (max ≤ right) trừ đếm (max ≤ left − 1)** → **một lượt với hai chỉ số "vị trí gần nhất"** → **đếm đóng góp của từng phần tử làm "max" bằng ngăn xếp đơn điệu** → **quy hoạch động theo điểm kết thúc**.
4. Hiểu mấu chốt: một phần tử **lớn hơn `right`** là "bức tường" không mảng con hợp lệ nào đi qua được; một phần tử **nhỏ hơn `left`** thì **không làm hỏng** mảng con (miễn là trong mảng con còn có phần tử thuộc `[left, right]`).
5. Tránh các lỗi kinh điển: chỉ đếm mảng con mà **mọi** phần tử thuộc `[left, right]`, coi phần tử nhỏ hơn `left` là bức tường, nhầm `left` với `left − 1` khi bù trừ, đếm cả mảng con **không** chứa phần tử nào thuộc đoạn, đếm trùng khi có phần tử bằng nhau ở cách ngăn xếp, và `O(n²)` với `n = 10^5`.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` và hai số nguyên `left`, `right`. Trả về **số lượng mảng con liên tiếp, không rỗng** sao cho **giá trị lớn nhất** của các phần tử trong mảng con đó nằm trong đoạn `[left, right]` (kể cả hai đầu mút).

Các test được tạo sao cho đáp án vừa trong số nguyên 32 bit.

**Ví dụ 1** (của đề)

- Input: `nums = [2,1,4,3]`, `left = 2`, `right = 3`
- Output: `3`
- Giải thích: ba mảng con thỏa: `[2]`, `[2, 1]`, `[3]`.

**Ví dụ 2** (của đề)

- Input: `nums = [2,9,2,5,6]`, `left = 2`, `right = 8`
- Output: `7`

**Ví dụ 3 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `nums = [1]`, `left = 2`, `right = 3`
- Output: `0`
- Giải thích: mảng con duy nhất `[1]` có max là `1`, không thuộc `[2, 3]`.

**Ví dụ 4 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `nums = [3,1,2,1]`, `left = 1`, `right = 2`
- Output: `6`
- Giải thích: các mảng con thỏa: `[1]` (vị trí 1), `[1,2]`, `[1,2,1]`, `[2]`, `[2,1]`, `[1]` (vị trí 3). Mọi mảng con chứa `3` có max là `3 > right` nên bị loại.

**Ràng buộc:**

- `1 <= nums.length <= 10^5`
- `0 <= nums[i] <= 10^9`
- `0 <= left <= right <= 10^9`

In [1]:
from typing import List
import random
import time

examples = [
    ([2, 1, 4, 3], 2, 3, 3),
    ([2, 9, 2, 5, 6], 2, 8, 7),
    ([1], 2, 3, 0),
    ([3, 1, 2, 1], 1, 2, 6),
]
for nums, left, right, expected in examples:
    print(nums, "left =", left, "right =", right, "-> kỳ vọng", expected)

[2, 1, 4, 3] left = 2 right = 3 -> kỳ vọng 3
[2, 9, 2, 5, 6] left = 2 right = 8 -> kỳ vọng 7
[1] left = 2 right = 3 -> kỳ vọng 0
[3, 1, 2, 1] left = 1 right = 2 -> kỳ vọng 6


## 3. Kiến thức nền

### 3.1. Mảng con liên tiếp (subarray) và số lượng của chúng

Một **mảng con** là một đoạn liên tiếp `nums[i..j]` với `0 ≤ i ≤ j < n`. Số mảng con của mảng dài `n` là `n(n + 1)/2` (với mỗi `j`, có `j + 1` điểm bắt đầu `i`). Đừng nhầm với **dãy con** (subsequence) — được bỏ phần tử ở giữa.

In [2]:
nums = [2, 1, 4, 3]
subarrays = [nums[i:j + 1] for i in range(len(nums)) for j in range(i, len(nums))]
print(subarrays)
print(len(subarrays), "=", len(nums) * (len(nums) + 1) // 2)

[[2], [2, 1], [2, 1, 4], [2, 1, 4, 3], [1], [1, 4], [1, 4, 3], [4], [4, 3], [3]]
10 = 10


### 3.2. Max chạy (running max)

Khi mở rộng mảng con sang phải một phần tử, max mới là `max(max_cũ, phần_tử_mới)`; không cần quét lại toàn bộ. Nhờ vậy duyệt mọi mảng con bắt đầu tại `i` tốn `O(n)` thay vì `O(n²)` (cho một `i`).

In [3]:
nums = [2, 1, 4, 3]
for i in range(len(nums)):
    cur = -1
    row = []
    for j in range(i, len(nums)):
        cur = max(cur, nums[j])
        row.append(cur)
    print(f"bắt đầu tại {i}: max của nums[{i}..j] với j = {i}.. là {row}")

bắt đầu tại 0: max của nums[0..j] với j = 0.. là [2, 2, 4, 4]
bắt đầu tại 1: max của nums[1..j] với j = 1.. là [1, 4, 4]
bắt đầu tại 2: max của nums[2..j] với j = 2.. là [4, 4]
bắt đầu tại 3: max của nums[3..j] với j = 3.. là [3]


### 3.3. Đếm mảng con bằng công thức `L(L + 1)/2`

Nếu một **đoạn liên tiếp** dài `L` thỏa một tính chất "kế thừa" (mọi mảng con của nó cũng thỏa), thì số mảng con của nó là `1 + 2 + ... + L = L(L + 1)/2`. Ta sẽ dùng cho tính chất "mọi phần tử `≤ bound`": mỗi đoạn tối đa liên tiếp các phần tử `≤ bound` có `L(L + 1)/2` mảng con thỏa.

In [4]:
def count_all_le(nums, bound):
    """Số mảng con mà MỌI phần tử <= bound (tức max <= bound)."""
    total, run = 0, 0
    for x in nums:
        run = run + 1 if x <= bound else 0
        total += run              # run = số mảng con kết thúc tại vị trí này (đều thỏa)
    return total

print(count_all_le([2, 9, 2, 5, 6], 8))       # đoạn [2] và [2,5,6]: 1 + 6 = 7
print(3 * 4 // 2)                              # 6 = số mảng con của đoạn dài 3

7
6


### 3.4. "Bức tường" và phần tử nhỏ

Với đoạn `[left, right]` cố định, ta chia các phần tử thành **ba loại**:

- **nhỏ**: `x < left` — không đủ để làm max hợp lệ, nhưng cũng **không làm hỏng** nếu mảng con còn chứa một phần tử "vừa";
- **vừa**: `left ≤ x ≤ right` — mảng con hợp lệ khi **max thuộc loại vừa**;
- **lớn**: `x > right` — **bức tường**: mảng con nào chứa nó có max `> right` ⇒ bị loại.

Mảng con hợp lệ ⇔ **không chứa phần tử lớn** và **chứa ít nhất một phần tử vừa**.

In [5]:
nums, left, right = [3, 1, 2, 1], 1, 2
kind = ["lớn" if x > right else "nhỏ" if x < left else "vừa" for x in nums]
print(list(zip(nums, kind)))

[(3, 'lớn'), (1, 'vừa'), (2, 'vừa'), (1, 'vừa')]


### 3.5. Hai chỉ số "vị trí gần nhất"

Khi duyệt `i` từ trái sang phải, ta nhớ hai số: `last_big` = **vị trí gần nhất** của một phần tử **lớn** (`> right`), `last_ok` = vị trí gần nhất của một phần tử **vừa** (`left ≤ x ≤ right`). (Khởi tạo `-1` nghĩa là "chưa thấy".) Số mảng con **kết thúc tại `i`** và hợp lệ là `last_ok - last_big` nếu `last_ok > last_big`, ngược lại `0` — sẽ chứng minh ở mục 10.

### 3.6. Ngăn xếp đơn điệu (monotonic stack) tìm "lớn hơn gần nhất"

Với mỗi vị trí `i`, ta muốn biết **vị trí gần nhất bên trái có giá trị `≥ nums[i]`** (`prev[i]`) và **vị trí gần nhất bên phải có giá trị `> nums[i]`** (`nxt[i]`). Duyệt một lượt, giữ ngăn xếp các chỉ số có giá trị **giảm dần**; khi gặp phần tử mới, **pop** các phần tử nhỏ hơn rồi đọc đỉnh còn lại. Mỗi phần tử vào và ra đúng một lần nên tổng `O(n)`.

In [6]:
def prev_ge(nums):
    """prev[i] = chỉ số gần nhất bên trái có nums[k] >= nums[i], hoặc -1."""
    stack, prev = [], [-1] * len(nums)
    for i, x in enumerate(nums):
        while stack and nums[stack[-1]] < x:
            stack.pop()
        prev[i] = stack[-1] if stack else -1
        stack.append(i)
    return prev

def next_gt(nums):
    """nxt[i] = chỉ số gần nhất bên phải có nums[k] > nums[i], hoặc n."""
    n = len(nums)
    stack, nxt = [], [n] * n
    for i in range(n - 1, -1, -1):
        while stack and nums[stack[-1]] <= nums[i]:
            stack.pop()
        nxt[i] = stack[-1] if stack else n
        stack.append(i)
    return nxt

nums = [2, 1, 4, 3]
print(prev_ge(nums), next_gt(nums))        # [-1, 0, -1, 2] [2, 2, 4, 4]

[-1, 0, -1, 2] [2, 2, 4, 4]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Xét `nums = [2, 9, 2, 5, 6]`, `left = 2`, `right = 8` (đáp án `7`).

Phần tử `9 > 8` là bức tường, chia mảng thành hai đoạn **không có phần tử lớn**: `[2]` (vị trí 0) và `[2, 5, 6]` (vị trí 2–4). Mọi mảng con hợp lệ nằm gọn trong một trong hai đoạn.

- Đoạn `[2]`: có 1 mảng con `[2]`, max `2 ∈ [2, 8]` ✓ → 1.
- Đoạn `[2, 5, 6]`: có `3 · 4 / 2 = 6` mảng con, max đều thuộc `[2, 6]` ⊂ `[2, 8]` ✓ → 6.

Tổng `1 + 6 = 7`. ✓.

Nếu có thêm phần tử `< left` (ví dụ `left = 3`): thì trong từng đoạn, ta phải loại các mảng con mà **mọi phần tử đều `< left`** (max `< left`). Đó là ý "bù trừ": **(số mảng con với max `≤ right`) − (số mảng con với max `≤ left - 1`)**.

Thử với ví dụ 1: `nums = [2,1,4,3]`, `left = 2`, `right = 3`:

- max `≤ 3`: các đoạn `[2, 1]` (3 mảng con) và `[3]` (1 mảng con) → `4`.
- max `≤ 1` (tức `≤ left - 1`): đoạn `[1]` → `1`.
- Đáp án `4 - 1 = 3` ✓ (`[2]`, `[2,1]`, `[3]`).

Cell dưới chạy lại đúng phép tính này.

In [7]:
def count_le(nums, bound):
    total = run = 0
    for x in nums:
        run = run + 1 if x <= bound else 0
        total += run
    return total

for nums, left, right, expected in examples:
    a, b = count_le(nums, right), count_le(nums, left - 1)
    print(f"{nums}: count(max <= {right}) = {a}, count(max <= {left - 1}) = {b} -> {a - b}")
    assert a - b == expected

[2, 1, 4, 3]: count(max <= 3) = 4, count(max <= 1) = 1 -> 3
[2, 9, 2, 5, 6]: count(max <= 8) = 7, count(max <= 1) = 0 -> 7
[1]: count(max <= 3) = 1, count(max <= 1) = 1 -> 0
[3, 1, 2, 1]: count(max <= 2) = 6, count(max <= 0) = 0 -> 6


## 5. Cách 1 — Brute Force / baseline: thử mọi mảng con

**Ý tưởng.** Với mỗi điểm bắt đầu `i`, mở rộng điểm kết thúc `j` sang phải, giữ **max chạy** `cur`; mỗi mảng con `nums[i..j]` có max `cur` — nếu `left ≤ cur ≤ right` thì đếm. Có thể **dừng sớm** khi `cur > right` vì mọi mở rộng sau đó vẫn có max `> right`.

In [8]:
def count_brute(nums, left, right):
    n = len(nums)
    total = 0
    for i in range(n):
        cur = -1
        for j in range(i, n):
            cur = max(cur, nums[j])
            if cur > right:
                break                       # mọi mảng con dài hơn cũng có max > right
            if cur >= left:
                total += 1
    return total


for nums, left, right, expected in examples:
    got = count_brute(nums, left, right)
    print(nums, "->", got)
    assert got == expected

[2, 1, 4, 3] -> 3
[2, 9, 2, 5, 6] -> 7
[1] -> 0
[3, 1, 2, 1] -> 6


**Độ phức tạp.** Thời gian **O(n²)** trong trường hợp xấu (nhiều phần tử `≤ right`): với `n = 10^5` là khoảng `5 × 10^9` phép — **TLE**. Bộ nhớ **O(1)**.

**Nút thắt.** Ta đếm **từng** mảng con một, trong khi nhiều mảng con có thể được đếm **gộp** nhờ công thức `L(L + 1)/2` hoặc nhờ "có bao nhiêu điểm bắt đầu hợp lệ cho một điểm kết thúc" (một phép trừ hai chỉ số).

## 6. Ý tưởng thuật toán / bước đột phá

Có **hai cách nhìn gộp** dẫn tới `O(n)`:

**A. Bù trừ (complement).** Điều kiện "max ∈ `[left, right]`" = "max `≤ right`" **và không** "max `≤ left - 1`". Vì tập "max `≤ left - 1`" **nằm trọn trong** tập "max `≤ right`" nên:

```
đáp án = f(right) - f(left - 1)    với    f(b) = số mảng con có max ≤ b
```

và `f(b)` tính trong một lượt: mỗi đoạn liên tiếp các phần tử `≤ b` dài `L` đóng góp `L(L + 1)/2` (hoặc: duy trì `run` = độ dài đoạn kết thúc tại vị trí hiện tại, cộng `run` vào tổng).

**B. Đếm theo điểm kết thúc.** Cố định điểm kết thúc `i`. Mảng con `nums[s..i]` hợp lệ ⇔ **không chứa phần tử lớn** (`s > last_big`) và **chứa ít nhất một phần tử vừa** (`s ≤ last_ok`). Vậy số điểm bắt đầu hợp lệ là `last_ok - last_big` (nếu dương). Cộng qua mọi `i`.

Từ B, ta có thêm hai biến thể: **DP** (`dp[i]` = số mảng con hợp lệ kết thúc tại `i`) và **đóng góp của từng "max"** bằng ngăn xếp đơn điệu.

## 7. Các cách tiếp theo

Tổng cộng 5 cách: cách 1 (vét cạn, mục 5) và bốn cách dưới đây.

### 7.1. Cách 2 — Bù trừ: `f(right) - f(left - 1)`

**Giải thích từng đoạn.**

- `f(b)`: duyệt `nums`; `run` = độ dài đoạn liên tiếp `≤ b` kết thúc tại vị trí hiện tại (về `0` nếu phần tử `> b`); cộng `run` vào tổng (có đúng `run` mảng con kết thúc tại đây mà mọi phần tử `≤ b`).
- Đáp án `f(right) - f(left - 1)`.

Với `left = 0` thì `left - 1 = -1`, và `f(-1) = 0` (không phần tử nào `≤ -1` vì mọi phần tử `≥ 0`), công thức vẫn đúng.

In [9]:
def count_complement(nums, left, right, verbose=False):
    def f(bound):
        total = run = 0
        for x in nums:
            run = run + 1 if x <= bound else 0
            total += run
        return total

    a, b = f(right), f(left - 1)
    if verbose:
        print(f"  f({right}) = {a}, f({left - 1}) = {b}")
    return a - b


for nums, left, right, expected in examples:
    got = count_complement(nums, left, right, verbose=(nums == examples[0][0]))
    assert got == expected
print("OK")

  f(3) = 4, f(1) = 1
OK


**Code nộp LeetCode (cách 2):**

In [10]:
class Solution2:
    def numSubarrayBoundedMax(self, nums: List[int], left: int, right: int) -> int:
        def count(bound: int) -> int:
            total = run = 0
            for x in nums:
                run = run + 1 if x <= bound else 0
                total += run
            return total

        return count(right) - count(left - 1)


for nums, left, right, expected in examples:
    assert Solution2().numSubarrayBoundedMax(nums, left, right) == expected
print("Solution2 OK")

Solution2 OK


**Độ phức tạp cách 2.** Hai lượt duyệt: thời gian **O(n)**, bộ nhớ **O(1)**.

### 7.2. Cách 3 — Một lượt với hai chỉ số `last_big` và `last_ok`

**Ý tưởng.** Duyệt `i = 0..n-1`, giữ:

- `last_big`: vị trí gần nhất của phần tử `> right` (khởi tạo `-1`);
- `last_ok`: vị trí gần nhất của phần tử `∈ [left, right]` (khởi tạo `-1`).

Tại mỗi `i`: nếu `nums[i] > right` thì `last_big = i`; nếu `left ≤ nums[i] ≤ right` thì `last_ok = i`. Số mảng con hợp lệ **kết thúc tại `i`** là **`max(0, last_ok - last_big)`**, cộng vào đáp án. (Điều kiện `last_ok > last_big` đã nằm trong `max(0, ·)`.)

In [11]:
def count_pointers(nums, left, right, verbose=False):
    last_big = last_ok = -1
    total = 0
    for i, x in enumerate(nums):
        if x > right:
            last_big = i
        elif x >= left:
            last_ok = i
        add = max(0, last_ok - last_big)
        total += add
        if verbose:
            print(f"  i = {i}, x = {x}: last_big = {last_big}, last_ok = {last_ok} -> +{add}, total = {total}")
    return total


for nums, left, right, expected in examples:
    got = count_pointers(nums, left, right, verbose=(nums == examples[0][0]))
    assert got == expected
print("OK")

  i = 0, x = 2: last_big = -1, last_ok = 0 -> +1, total = 1
  i = 1, x = 1: last_big = -1, last_ok = 0 -> +1, total = 2
  i = 2, x = 4: last_big = 2, last_ok = 0 -> +0, total = 2
  i = 3, x = 3: last_big = 2, last_ok = 3 -> +1, total = 3
OK


**Code nộp LeetCode (cách 3):**

In [12]:
class Solution3:
    def numSubarrayBoundedMax(self, nums: List[int], left: int, right: int) -> int:
        ans = 0
        last_big = last_ok = -1
        for i, x in enumerate(nums):
            if x > right:
                last_big = i
            elif x >= left:
                last_ok = i
            if last_ok > last_big:
                ans += last_ok - last_big
        return ans


for nums, left, right, expected in examples:
    assert Solution3().numSubarrayBoundedMax(nums, left, right) == expected
print("Solution3 OK")

Solution3 OK


**Độ phức tạp cách 3.** Một lượt: thời gian **O(n)**, bộ nhớ **O(1)**.

### 7.3. Cách 4 — Đóng góp của mỗi phần tử làm "max" (ngăn xếp đơn điệu)

**Ý tưởng.** Với mỗi mảng con, chọn **một đại diện duy nhất** cho phần tử max (để không đếm trùng khi có nhiều phần tử bằng nhau): lấy phần tử max **ngoài cùng bên trái**. Định nghĩa:

- `prev[i]` = vị trí gần nhất bên trái có `nums[k] ≥ nums[i]` (hoặc `-1`);
- `nxt[i]` = vị trí gần nhất bên phải có `nums[k] > nums[i]` (hoặc `n`).

Khi đó `nums[i]` là phần tử max trái nhất của đúng các mảng con `nums[s..e]` với `prev[i] < s ≤ i ≤ e < nxt[i]`: có `(i - prev[i]) × (nxt[i] - i)` mảng con. Mỗi mảng con được gán cho **đúng một** `i`.

Đáp án = tổng `(i - prev[i]) × (nxt[i] - i)` trên các `i` có `left ≤ nums[i] ≤ right`.

In [13]:
def count_stack(nums, left, right, verbose=False):
    n = len(nums)
    prev = [-1] * n
    stack = []
    for i, x in enumerate(nums):                       # gần nhất bên trái có giá trị >= x
        while stack and nums[stack[-1]] < x:
            stack.pop()
        prev[i] = stack[-1] if stack else -1
        stack.append(i)
    nxt = [n] * n
    stack = []
    for i in range(n - 1, -1, -1):                     # gần nhất bên phải có giá trị > x
        while stack and nums[stack[-1]] <= nums[i]:
            stack.pop()
        nxt[i] = stack[-1] if stack else n
        stack.append(i)
    total = 0
    for i, x in enumerate(nums):
        if left <= x <= right:
            cnt = (i - prev[i]) * (nxt[i] - i)
            total += cnt
            if verbose:
                print(f"  i = {i}, x = {x}: prev = {prev[i]}, nxt = {nxt[i]} -> {i - prev[i]} * {nxt[i] - i} = {cnt}")
    return total


for nums, left, right, expected in examples:
    got = count_stack(nums, left, right, verbose=(nums == examples[3][0]))
    assert got == expected
print("OK")

  i = 1, x = 1: prev = 0, nxt = 2 -> 1 * 1 = 1
  i = 2, x = 2: prev = 0, nxt = 4 -> 2 * 2 = 4
  i = 3, x = 1: prev = 2, nxt = 4 -> 1 * 1 = 1
OK


**Code nộp LeetCode (cách 4):**

In [14]:
class Solution4:
    def numSubarrayBoundedMax(self, nums: List[int], left: int, right: int) -> int:
        n = len(nums)
        prev, nxt = [-1] * n, [n] * n
        stack: List[int] = []
        for i, x in enumerate(nums):
            while stack and nums[stack[-1]] < x:
                stack.pop()
            prev[i] = stack[-1] if stack else -1
            stack.append(i)
        stack = []
        for i in range(n - 1, -1, -1):
            while stack and nums[stack[-1]] <= nums[i]:
                stack.pop()
            nxt[i] = stack[-1] if stack else n
            stack.append(i)
        return sum((i - prev[i]) * (nxt[i] - i) for i, x in enumerate(nums) if left <= x <= right)


for nums, left, right, expected in examples:
    assert Solution4().numSubarrayBoundedMax(nums, left, right) == expected
print("Solution4 OK")

Solution4 OK


**Độ phức tạp cách 4.** Hai lượt ngăn xếp (mỗi phần tử vào/ra một lần) và một lượt cộng: thời gian **O(n)**, bộ nhớ **O(n)** (hai mảng `prev`, `nxt` và ngăn xếp). Cách này dài hơn nhưng **mạnh và tổng quát**: cùng khung dùng cho *Sum of Subarray Minimums (907)*, *Sum of Subarray Ranges (2104)*.

### 7.4. Cách 5 — Quy hoạch động theo điểm kết thúc

**Ý tưởng.** Gọi `dp[i]` là **số mảng con hợp lệ kết thúc tại `i`**. Xét `x = nums[i]`:

- `x > right`: mọi mảng con kết thúc ở `i` có max `≥ x > right` → `dp[i] = 0`.
- `left ≤ x ≤ right`: mảng con `nums[s..i]` có max đúng bằng... hợp lệ **với mọi `s` sau bức tường gần nhất** (vì có `x` thuộc đoạn và không có phần tử lớn) → `dp[i] = i - last_big` (số điểm bắt đầu `s` từ `last_big + 1` đến `i`).
- `x < left`: mảng con kết thúc ở `i` hợp lệ ⇔ mảng con `nums[s..i-1]` hợp lệ (thêm `x` nhỏ không đổi max; với `s = i` mảng con `[x]` có max `x < left` nên không hợp lệ) → `dp[i] = dp[i-1]`.

Đáp án `sum(dp)`.

In [15]:
def count_dp(nums, left, right, verbose=False):
    n = len(nums)
    dp = [0] * n
    last_big = -1
    for i, x in enumerate(nums):
        if x > right:
            last_big = i
            dp[i] = 0
        elif x >= left:
            dp[i] = i - last_big
        else:
            dp[i] = dp[i - 1] if i > 0 else 0
    if verbose:
        print("  dp =", dp)
    return sum(dp)


for nums, left, right, expected in examples:
    got = count_dp(nums, left, right, verbose=(nums == examples[1][0]))
    assert got == expected
print("OK")

  dp = [1, 0, 1, 2, 3]
OK


**Code nộp LeetCode (cách 5):**

In [16]:
class Solution5:
    def numSubarrayBoundedMax(self, nums: List[int], left: int, right: int) -> int:
        ans = cur = 0               # cur = dp[i-1] (số mảng con hợp lệ kết thúc ở vị trí trước)
        last_big = -1
        for i, x in enumerate(nums):
            if x > right:
                last_big = i
                cur = 0
            elif x >= left:
                cur = i - last_big
            # x < left: cur giữ nguyên (= dp[i-1])
            ans += cur
        return ans


for nums, left, right, expected in examples:
    assert Solution5().numSubarrayBoundedMax(nums, left, right) == expected
print("Solution5 OK")

Solution5 OK


**Độ phức tạp cách 5.** Một lượt: thời gian **O(n)**, bộ nhớ **O(1)** (chỉ giữ `cur`, `last_big`, `ans`). Về bản chất đây là cách 3 viết theo ngôn ngữ DP: `dp[i] = last_ok - last_big` và ta chỉ cập nhật `dp[i]` bằng cách khác (`i - last_big` khi vừa, giữ nguyên khi nhỏ).

## 8. TRACE từng bước

Ta trace trên `nums = [2, 1, 4, 3]`, `left = 2`, `right = 3` (đáp án `3`) và `[2, 9, 2, 5, 6]`, `left = 2`, `right = 8` (đáp án `7`).

### 8.1. Trace cách 2 (bù trừ)

Bảng trạng thái cho hàm `f(bound)`:

| Cột | Ý nghĩa |
|---|---|
| `x` | phần tử đang xét |
| `x <= bound?` | có nối tiếp đoạn liên tiếp `≤ bound` không |
| `run` | độ dài đoạn liên tiếp `≤ bound` kết thúc tại đây |
| `total` | tổng số mảng con có max `≤ bound` tới đây |

In [17]:
def trace_f(nums, bound):
    print(f"f({bound}):")
    total = run = 0
    for i, x in enumerate(nums):
        ok = x <= bound
        run = run + 1 if ok else 0
        total += run
        print(f"  i = {i} x = {x:>2} | x <= {bound}? {str(ok):<5} | run = {run} | total = {total}")
    return total

def trace_complement(nums, left, right):
    a = trace_f(nums, right)
    b = trace_f(nums, left - 1)
    print(f"đáp án = f({right}) - f({left - 1}) = {a} - {b} = {a - b}")
    return a - b

trace_complement([2, 1, 4, 3], 2, 3)

f(3):
  i = 0 x =  2 | x <= 3? True  | run = 1 | total = 1
  i = 1 x =  1 | x <= 3? True  | run = 2 | total = 3
  i = 2 x =  4 | x <= 3? False | run = 0 | total = 3
  i = 3 x =  3 | x <= 3? True  | run = 1 | total = 4
f(1):
  i = 0 x =  2 | x <= 1? False | run = 0 | total = 0
  i = 1 x =  1 | x <= 1? True  | run = 1 | total = 1
  i = 2 x =  4 | x <= 1? False | run = 0 | total = 1
  i = 3 x =  3 | x <= 1? False | run = 0 | total = 1
đáp án = f(3) - f(1) = 4 - 1 = 3


3

Quan sát: với `f(3)`: `run` tăng `1, 2` ở hai phần tử đầu (mảng con `[2]`, `[1]`, `[2,1]` → 3), về `0` ở `4`, rồi `1` ở `3` → tổng `4`. Với `f(1)`: chỉ phần tử `1` thỏa → tổng `1`. Hiệu `3`.

### 8.2. Trace cách 3 (hai chỉ số `last_big`, `last_ok`) trên ví dụ 2

In [18]:
def trace_pointers(nums, left, right):
    last_big = last_ok = -1
    total = 0
    print(f"left = {left}, right = {right}")
    for i, x in enumerate(nums):
        kind = "lớn (tường)" if x > right else "vừa" if x >= left else "nhỏ"
        if x > right:
            last_big = i
        elif x >= left:
            last_ok = i
        add = max(0, last_ok - last_big)
        total += add
        print(f"i = {i} x = {x}: {kind:<12} last_big = {last_big:>2} last_ok = {last_ok:>2} -> +{add} (tổng {total})")
    return total

print("kết quả:", trace_pointers([2, 9, 2, 5, 6], 2, 8))

left = 2, right = 8
i = 0 x = 2: vừa          last_big = -1 last_ok =  0 -> +1 (tổng 1)
i = 1 x = 9: lớn (tường)  last_big =  1 last_ok =  0 -> +0 (tổng 1)
i = 2 x = 2: vừa          last_big =  1 last_ok =  2 -> +1 (tổng 2)
i = 3 x = 5: vừa          last_big =  1 last_ok =  3 -> +2 (tổng 4)
i = 4 x = 6: vừa          last_big =  1 last_ok =  4 -> +3 (tổng 7)
kết quả: 7


Đọc trace: ở `i = 0` (`x = 2` vừa) có `last_ok = 0`, `last_big = -1` nên `+1` (mảng con `[2]`). Ở `i = 1` (`x = 9` lớn) tường dời tới vị trí 1, `last_ok - last_big = 0 - 1 < 0` nên `+0`: mọi mảng con kết thúc ở đây chứa `9`. Ở `i = 2` (`x = 2`): `last_ok = 2` nên `+ (2 - 1) = 1` (`[2]`). Ở `i = 3` (`5`): `+2` (`[5]`, `[2,5]`). Ở `i = 4` (`6`): `+3`. Tổng `1 + 0 + 1 + 2 + 3 = 7`.

### 8.3. Trace cách 4 (ngăn xếp đơn điệu) trên ví dụ 4: `[3, 1, 2, 1]`, `left = 1`, `right = 2`

In `prev`, `nxt` rồi đóng góp từng phần tử:

In [19]:
def trace_stack(nums, left, right):
    n = len(nums)
    prev, nxt = prev_ge(nums), next_gt(nums)
    print("nums:", nums)
    print("prev (gần nhất bên trái >= ):", prev)
    print("nxt  (gần nhất bên phải >  ):", nxt)
    total = 0
    for i, x in enumerate(nums):
        in_range = left <= x <= right
        cnt = (i - prev[i]) * (nxt[i] - i) if in_range else 0
        total += cnt
        note = f"{i - prev[i]} * {nxt[i] - i} = {cnt}" if in_range else "ngoài [left, right] -> bỏ"
        print(f"i = {i} x = {x}: {note}")
    return total

print("kết quả:", trace_stack([3, 1, 2, 1], 1, 2))

nums: [3, 1, 2, 1]
prev (gần nhất bên trái >= ): [-1, 0, 0, 2]
nxt  (gần nhất bên phải >  ): [4, 2, 4, 4]
i = 0 x = 3: ngoài [left, right] -> bỏ
i = 1 x = 1: 1 * 1 = 1
i = 2 x = 2: 2 * 2 = 4
i = 3 x = 1: 1 * 1 = 1
kết quả: 6


Đọc trace: phần tử `3` (`> right`) bị bỏ. Phần tử `1` ở vị trí 1: bên trái gần nhất `≥ 1` là `3` (vị trí 0), bên phải gần nhất `> 1` là `2` (vị trí 2) ⇒ `(1 - 0) × (2 - 1) = 1` mảng con (`[1]`). Phần tử `2` ở vị trí 2: `prev = 0` (giá trị `3 ≥ 2`), `nxt = 4` (hết) ⇒ `(2 - 0) × (4 - 2) = 4` mảng con (`[2]`, `[1,2]`, `[2,1]`, `[1,2,1]`). Phần tử `1` ở vị trí 3: `prev = 2` (giá trị `2 ≥ 1`), `nxt = 4` ⇒ `(3 - 2) × (4 - 3) = 1` (`[1]`). Tổng `1 + 4 + 1 = 6`.

### 8.4. Trace cách 5 (DP) trên ví dụ 1

In [20]:
def trace_dp(nums, left, right):
    last_big, cur, total = -1, 0, 0
    for i, x in enumerate(nums):
        if x > right:
            last_big, cur, why = i, 0, "x > right -> dp = 0, đặt tường"
        elif x >= left:
            cur, why = i - last_big, f"vừa -> dp = i - last_big = {i} - {last_big}"
        else:
            why = "nhỏ -> dp giữ nguyên dp[i-1]"
        total += cur
        print(f"i = {i} x = {x}: {why:<38} dp[{i}] = {cur} | tổng = {total}")
    return total

print("kết quả:", trace_dp([2, 1, 4, 3], 2, 3))

i = 0 x = 2: vừa -> dp = i - last_big = 0 - -1      dp[0] = 1 | tổng = 1
i = 1 x = 1: nhỏ -> dp giữ nguyên dp[i-1]           dp[1] = 1 | tổng = 2
i = 2 x = 4: x > right -> dp = 0, đặt tường         dp[2] = 0 | tổng = 2
i = 3 x = 3: vừa -> dp = i - last_big = 3 - 2       dp[3] = 1 | tổng = 3
kết quả: 3


Đọc trace: `i = 0` (`2` vừa): `dp = 0 - (-1) = 1` (`[2]`). `i = 1` (`1` nhỏ): `dp = dp[0] = 1` (`[2, 1]` hợp lệ; `[1]` thì không). `i = 2` (`4` lớn): `dp = 0`, tường. `i = 3` (`3` vừa): `dp = 3 - 2 = 1` (`[3]`). Tổng `3`.

## 9. Code nộp LeetCode

Chọn **cách 3 (hai chỉ số `last_big`, `last_ok`)**: một lượt, `O(1)` bộ nhớ, ngắn và dễ chứng minh. Đây là code đúng class/method/signature (trên LeetCode `List` có sẵn; ở đây ta đã import ở trên):

In [21]:
class Solution:
    def numSubarrayBoundedMax(self, nums: List[int], left: int, right: int) -> int:
        ans = 0
        last_big = -1            # vị trí gần nhất của phần tử > right ("bức tường")
        last_ok = -1             # vị trí gần nhất của phần tử thuộc [left, right]
        for i, x in enumerate(nums):
            if x > right:
                last_big = i
            elif x >= left:
                last_ok = i
            if last_ok > last_big:
                ans += last_ok - last_big   # số điểm bắt đầu s với last_big < s <= last_ok
        return ans


for nums, left, right, expected in examples:
    got = Solution().numSubarrayBoundedMax(nums, left, right)
    print(nums, left, right, "->", got)
    assert got == expected

[2, 1, 4, 3] 2 3 -> 3
[2, 9, 2, 5, 6] 2 8 -> 7
[1] 2 3 -> 0
[3, 1, 2, 1] 1 2 -> 6


## 10. Vì sao thuật toán đúng

**Đặc trưng mảng con hợp lệ.** Mảng con `nums[s..e]` có max thuộc `[left, right]` ⇔ (a) **mọi** phần tử `≤ right` (max `≤ right`) **và** (b) **ít nhất một** phần tử `≥ left` (max `≥ left`). Vì max `≥ left` ⇔ có phần tử `≥ left`; kết hợp với (a) thì phần tử đó thuộc `[left, right]`. Vậy: **không chứa phần tử lớn** và **chứa phần tử vừa**.

**Cách 3 (hai chỉ số).** Cố định điểm kết thúc `e = i`. Gọi `B = last_big` (vị trí gần nhất `≤ i` của phần tử lớn, `-1` nếu chưa có) và `K = last_ok` (vị trí gần nhất `≤ i` của phần tử vừa).

- (a) ⇔ không có phần tử lớn trong `[s, i]` ⇔ `s > B` (vì `B` là phần tử lớn gần nhất; mọi phần tử lớn khác ở xa hơn bên trái `B`).
- (b), kết hợp với (a): phải có phần tử vừa trong `[s, i]`; phần tử vừa gần nhất là `K`, nên điều kiện là `s ≤ K` (nếu `K ≤ B` thì không có phần tử vừa nào nằm sau `B`, tức mọi `s > B` cho `s > K`, không có phần tử vừa → vô nghiệm).
- Vậy `s` hợp lệ ⇔ `B < s ≤ K`, có `K - B` giá trị nếu `K > B`, ngược lại `0`.

Cộng qua `i = 0..n-1` ta được tổng số mảng con hợp lệ (mỗi mảng con được đếm đúng một lần, theo điểm kết thúc của nó).

**Cách 2 (bù trừ).** Tập `A = {max ≤ right}` chứa tập `C = {max ≤ left - 1}` (vì `left - 1 < right`... luôn đúng khi `left ≤ right`). Đáp án là `|A \ C| = |A| - |C|`. Và `f(b)` đếm đúng: mảng con có mọi phần tử `≤ b` ⇔ nằm gọn trong một đoạn tối đa các phần tử `≤ b`, và đoạn dài `L` chứa `L(L+1)/2` mảng con (`run` cộng dồn `1 + 2 + ... + L`).

**Cách 4 (ngăn xếp).** Mỗi mảng con có **phần tử max trái nhất** duy nhất (chỉ số `i`). `nums[i]` là max trái nhất của `nums[s..e]` ⇔ mọi phần tử bên trái trong mảng con `< nums[i]` (tức `s > prev[i]` với `prev[i]` là phần tử `≥ nums[i]` gần nhất bên trái) và mọi phần tử bên phải `≤ nums[i]` (tức `e < nxt[i]` với `nxt[i]` là phần tử `> nums[i]` gần nhất bên phải). Số mảng con là `(i - prev[i]) × (nxt[i] - i)`. Mảng con hợp lệ ⇔ max `nums[i] ∈ [left, right]`. Hai quy ước `≥` ở trái và `>` ở phải bảo đảm **không đếm trùng** khi có phần tử bằng nhau.

**Cách 5 (DP).** Là cách 3 viết dạng truy hồi: `x > right ⇒ 0`; `x` vừa `⇒ i - last_big` (mọi `s > last_big`, vì `nums[i]` vừa nằm trong mảng con); `x` nhỏ `⇒ dp[i-1]` (mảng con kết thúc ở `i` hợp lệ ⇔ `nums[s..i-1]` hợp lệ, vì thêm phần tử nhỏ không đổi max và `s = i` cho mảng con `[x]` không hợp lệ).

**Cách 1** đúng theo định nghĩa (dừng sớm khi `cur > right` là hợp lệ vì max chỉ tăng).

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Thử mọi mảng con | O(n²) | O(1) | `n = 10^5` → TLE |
| 2. Bù trừ `f(right) - f(left - 1)` | O(n) | O(1) | hai lượt, rất dễ nhớ |
| 3. Hai chỉ số `last_big`, `last_ok` | O(n) | O(1) | **bản nộp**, một lượt |
| 4. Ngăn xếp đơn điệu, đóng góp mỗi max | O(n) | O(n) | khung tổng quát cho họ bài "tổng theo max/min" |
| 5. DP theo điểm kết thúc | O(n) | O(1) | cách 3 dạng truy hồi |

**Chi tiết cách 4.** Mỗi phần tử được push và pop tối đa một lần ở mỗi trong hai lượt, nên `O(n)` dù có vòng `while` lồng.

**Chú ý về số lớn.** Với `n = 10^5` số mảng con tối đa là `~5 × 10^9`, vượt `int` 32 bit; đề bảo đảm test được chọn để đáp án vừa 32 bit. Python không tràn số nên không phải lo.

## 12. Bộ test

Ta kiểm tra theo nhiều lớp:

1. Ví dụ của đề (và các ví dụ tự đặt) cho cả 5 cách + bản nộp.
2. Edge case: `n = 1`, `left = 0`, `left == right`, toàn phần tử bằng nhau, toàn phần tử lớn, toàn phần tử nhỏ, mảng tăng/giảm dần, giá trị biên `10^9`.
3. **Phản ví dụ** của các cách làm sai.
4. Đối chiếu với **oracle độc lập** (với mỗi mảng con dùng `max(nums[i:j+1])` — `O(n³)`) trên input ngẫu nhiên có seed.
5. Test lớn `n = 10^5` và so sánh thời gian.

In [22]:
def oracle_cubic(nums, left, right):
    """Oracle độc lập: duyệt mọi mảng con, dùng max() trên lát cắt (O(n^3)), dành cho n nhỏ."""
    n = len(nums)
    return sum(1 for i in range(n) for j in range(i, n) if left <= max(nums[i:j + 1]) <= right)


for nums, left, right, expected in examples:
    assert oracle_cubic(nums, left, right) == expected
print("oracle khớp ví dụ")

solvers = {
    "brute": count_brute,
    "complement": count_complement,
    "pointers": count_pointers,
    "stack": count_stack,
    "dp": count_dp,
    "Solution2": lambda a, l, r: Solution2().numSubarrayBoundedMax(a, l, r),
    "Solution3": lambda a, l, r: Solution3().numSubarrayBoundedMax(a, l, r),
    "Solution4": lambda a, l, r: Solution4().numSubarrayBoundedMax(a, l, r),
    "Solution5": lambda a, l, r: Solution5().numSubarrayBoundedMax(a, l, r),
    "Solution": lambda a, l, r: Solution().numSubarrayBoundedMax(a, l, r),
}
for nums, left, right, expected in examples:
    for name, fn in solvers.items():
        assert fn(list(nums), left, right) == expected, (name, nums, left, right)
print("tất cả cách khớp ví dụ:", list(solvers))

oracle khớp ví dụ
tất cả cách khớp ví dụ: ['brute', 'complement', 'pointers', 'stack', 'dp', 'Solution2', 'Solution3', 'Solution4', 'Solution5', 'Solution']


In [23]:
# --- Edge case ---
edge_cases = [
    ([5], 5, 5),                               # n = 1, khớp đúng
    ([5], 6, 9),                               # n = 1, quá nhỏ
    ([5], 0, 4),                               # n = 1, quá lớn
    ([0], 0, 0),                               # toàn số 0, left = right = 0
    ([0, 0, 0], 0, 0),                         # toàn 0 -> 6
    ([2, 2, 2, 2], 2, 2),                      # toàn bằng nhau, left == right
    ([7, 8, 9], 1, 5),                         # toàn phần tử lớn -> 0
    ([1, 2, 3], 10, 20),                       # toàn phần tử nhỏ -> 0
    ([1, 2, 3, 4, 5], 2, 4),                   # tăng dần
    ([5, 4, 3, 2, 1], 2, 4),                   # giảm dần
    ([1, 1, 1, 1], 1, 1),                      # đều bằng nhau
    ([3, 1, 3, 1, 3], 3, 3),                   # nhiều max bằng nhau (tie)
    ([2, 1, 2, 1, 2], 2, 2),                   # tie xen kẽ phần tử nhỏ
    ([10**9, 0, 10**9], 0, 10**9),             # biên giá trị
    ([10**9, 5, 10**9], 5, 5),                 # phần tử lớn bao quanh
    ([4, 2, 4], 3, 3),                         # không có phần tử vừa -> 0
    ([2, 1, 4, 3], 0, 10),                     # left = 0, đoạn rộng
]
for nums, left, right in edge_cases:
    want = oracle_cubic(nums, left, right)
    for name, fn in solvers.items():
        got = fn(list(nums), left, right)
        assert got == want, (name, nums, left, right, got, want)
    print(nums, left, right, "->", want)

[5] 5 5 -> 1
[5] 6 9 -> 0
[5] 0 4 -> 0
[0] 0 0 -> 1
[0, 0, 0] 0 0 -> 6
[2, 2, 2, 2] 2 2 -> 10
[7, 8, 9] 1 5 -> 0
[1, 2, 3] 10 20 -> 0
[1, 2, 3, 4, 5] 2 4 -> 9
[5, 4, 3, 2, 1] 2 4 -> 9
[1, 1, 1, 1] 1 1 -> 10
[3, 1, 3, 1, 3] 3 3 -> 13
[2, 1, 2, 1, 2] 2 2 -> 13
[1000000000, 0, 1000000000] 0 1000000000 -> 6
[1000000000, 5, 1000000000] 5 5 -> 1
[4, 2, 4] 3 3 -> 0
[2, 1, 4, 3] 0 10 -> 10


**Phản ví dụ.** Các cách "trông có vẻ đúng" nhưng sai:

- *Sai A:* chỉ đếm mảng con mà **mọi** phần tử thuộc `[left, right]`.
- *Sai B:* coi phần tử `< left` là bức tường (reset như phần tử `> right`).
- *Sai C:* bù trừ bằng `f(right) - f(left)` (thiếu `- 1`).
- *Sai D:* hai chỉ số nhưng cộng `i - last_big` (không dùng `last_ok`), tức đếm cả mảng con **không có** phần tử vừa.
- *Sai E:* ngăn xếp đếm **trùng** khi có phần tử bằng nhau (dùng `≥` ở cả hai phía).

In [24]:
def wrong_all_in_range(nums, left, right):               # sai A
    total = run = 0
    for x in nums:
        run = run + 1 if left <= x <= right else 0
        total += run
    return total

def wrong_small_is_wall(nums, left, right):              # sai B
    total = run = 0
    for x in nums:
        if x > right or x < left:
            run = 0
        else:
            run += 1
        total += run
    return total

def wrong_complement(nums, left, right):                 # sai C
    def f(bound):
        total = run = 0
        for x in nums:
            run = run + 1 if x <= bound else 0
            total += run
        return total
    return f(right) - f(left)                            # thiếu - 1

def wrong_no_last_ok(nums, left, right):                 # sai D
    total, last_big = 0, -1
    for i, x in enumerate(nums):
        if x > right:
            last_big = i
        total += i - last_big                            # đếm cả mảng con toàn phần tử nhỏ
    return total

def wrong_stack_dup(nums, left, right):                  # sai E: cả hai phía đều "lớn hơn hẳn" -> đếm trùng khi bằng nhau
    n = len(nums)
    prev, stack = [-1] * n, []
    for i, x in enumerate(nums):
        while stack and nums[stack[-1]] <= x:            # SAI: prev cũng thành "lớn hơn hẳn" (phải là < để dừng ở phần tử bằng)
            stack.pop()
        prev[i] = stack[-1] if stack else -1
        stack.append(i)
    nxt, stack = [n] * n, []
    for i in range(n - 1, -1, -1):
        while stack and nums[stack[-1]] <= nums[i]:
            stack.pop()
        nxt[i] = stack[-1] if stack else n
        stack.append(i)
    return sum((i - prev[i]) * (nxt[i] - i) for i, x in enumerate(nums) if left <= x <= right)


tests = {
    "A": (wrong_all_in_range, [2, 1, 4, 3], 2, 3),
    "B": (wrong_small_is_wall, [2, 1, 4, 3], 2, 3),
    "C": (wrong_complement, [2, 1, 4, 3], 2, 3),
    "D": (wrong_no_last_ok, [1], 2, 3),
    "E": (wrong_stack_dup, [2, 2, 2], 2, 2),
}
for label, (fn, nums, left, right) in tests.items():
    truth = oracle_cubic(nums, left, right)
    print(f"sai {label}: {fn(nums, left, right):>3} | đúng: {truth} | nums = {nums}, [{left}, {right}]")
    assert fn(nums, left, right) != truth

sai A:   2 | đúng: 3 | nums = [2, 1, 4, 3], [2, 3]
sai B:   2 | đúng: 3 | nums = [2, 1, 4, 3], [2, 3]
sai C:   1 | đúng: 3 | nums = [2, 1, 4, 3], [2, 3]
sai D:   1 | đúng: 0 | nums = [1], [2, 3]
sai E:  10 | đúng: 6 | nums = [2, 2, 2], [2, 2]


Giải thích: **A** cho `2` (`[2]`, `[3]`) vì bỏ sót `[2, 1]` — phần tử `1` nhỏ hơn `left` nhưng đi kèm `2` thì max vẫn là `2`. **B** cũng ra `2` vì coi `1` là bức tường cắt đứt `[2, 1]`. **C** đếm cả các mảng con có max đúng `left` ở phần trừ nên **thiếu** chúng khỏi đáp án (ví dụ 1 ra `1` thay vì `3`). **D** cho `1` ở `[1]` với `left = 2`: đếm mảng con không có phần tử vừa. **E** đếm `[2,2,2]` quá nhiều vì mỗi phần tử bằng nhau đều tự coi mình là "max" của cùng mảng con.

**Đối chiếu nhiều input ngẫu nhiên (có seed, đúng ràng buộc).** Ta sinh `nums` với nhiều kiểu: giá trị nhỏ (nhiều phần tử bằng nhau), giá trị lớn tới `10^9`, và chọn `left`, `right` quanh các giá trị có thật trong `nums` để hay chạm biên.

In [25]:
def random_case(rng, n, vmax):
    nums = [rng.randint(0, vmax) for _ in range(n)]
    pool = nums + [0, vmax]
    a, b = rng.choice(pool), rng.choice(pool)
    left, right = min(a, b), max(a, b)
    return nums, left, right


rng = random.Random(795)
count = 0
for _ in range(800):
    n = rng.randint(1, 14)
    vmax = rng.choice([1, 2, 4, 10, 10**9])
    nums, left, right = random_case(rng, n, vmax)
    want = oracle_cubic(nums, left, right)
    for name, fn in solvers.items():
        original = list(nums)
        got = fn(original, left, right)
        assert got == want, (name, nums, left, right, got, want)
        assert original == nums, (name, "làm thay đổi input")
    count += 1
print("đối chiếu", count, "input ngẫu nhiên (n <= 14): tất cả cách khớp oracle O(n^3) và không sửa input")

# n trung bình: bỏ oracle O(n^3), dùng brute O(n^2) làm chuẩn
for _ in range(100):
    n = rng.randint(15, 300)
    vmax = rng.choice([2, 10, 100, 10**9])
    nums, left, right = random_case(rng, n, vmax)
    want = count_brute(nums, left, right)
    for name, fn in solvers.items():
        if name == "brute":
            continue
        assert fn(list(nums), left, right) == want, (name, n)
print("100 input n từ 15 đến 300: các cách O(n) khớp brute O(n^2)")

đối chiếu 800 input ngẫu nhiên (n <= 14): tất cả cách khớp oracle O(n^3) và không sửa input


100 input n từ 15 đến 300: các cách O(n) khớp brute O(n^2)


In [26]:
# Test lớn nhất theo ràng buộc: n = 10^5, so sánh thời gian
rng = random.Random(100000)
for vmax in (3, 1000, 10**9):
    nums = [rng.randint(0, vmax) for _ in range(10**5)]
    left, right = vmax // 3, (2 * vmax) // 3 + 1
    results = {}
    for name in ("complement", "pointers", "stack", "dp", "Solution2", "Solution3", "Solution4", "Solution5", "Solution"):
        t0 = time.perf_counter()
        results[name] = solvers[name](list(nums), left, right)
        dt = (time.perf_counter() - t0) * 1000
        if name in ("complement", "pointers", "stack", "dp"):
            print(f"vmax = {vmax:>10} | {name:>10}: {dt:7.1f} ms -> {results[name]}")
    assert len(set(results.values())) == 1, results

# trường hợp cực đại đáp án: toàn phần tử vừa -> n(n+1)/2
n = 10**5
nums = [5] * n
assert Solution().numSubarrayBoundedMax(nums, 5, 5) == n * (n + 1) // 2
assert Solution2().numSubarrayBoundedMax(nums, 0, 10) == n * (n + 1) // 2
assert Solution4().numSubarrayBoundedMax(nums, 5, 5) == n * (n + 1) // 2
print("toàn phần tử vừa, n = 10^5 ->", n * (n + 1) // 2, "(> 2^31 nhưng đề bảo đảm test thật vừa 32 bit)")

# brute O(n^2) trên mảng đồng đều, cỡ nhỏ hơn để thấy sự chậm
nums = [5] * 3000
t0 = time.perf_counter()
r = count_brute(nums, 5, 5)
print(f"brute trên n = 3000 toàn phần tử vừa: {(time.perf_counter() - t0) * 1000:.0f} ms -> {r}")
assert r == 3000 * 3001 // 2

vmax =          3 | complement:    19.4 ms -> 5000016984


vmax =          3 |   pointers:    24.2 ms -> 5000016984
vmax =          3 |      stack:    65.7 ms -> 5000016984


vmax =          3 |         dp:    16.6 ms -> 5000016984


vmax =       1000 | complement:    13.1 ms -> 150510


vmax =       1000 |   pointers:    13.4 ms -> 150510
vmax =       1000 |      stack:    42.4 ms -> 150510
vmax =       1000 |         dp:    11.8 ms -> 150510


vmax = 1000000000 | complement:    19.8 ms -> 152078


vmax = 1000000000 |   pointers:    13.4 ms -> 152078
vmax = 1000000000 |      stack:    48.9 ms -> 152078
vmax = 1000000000 |         dp:    13.2 ms -> 152078


toàn phần tử vừa, n = 10^5 -> 5000050000 (> 2^31 nhưng đề bảo đảm test thật vừa 32 bit)


brute trên n = 3000 toàn phần tử vừa: 362 ms -> 4501500


## 13. Các lỗi người mới hay gặp

### Lỗi 1: chỉ đếm mảng con mà **mọi** phần tử thuộc `[left, right]`

Đề hỏi về **max** của mảng con, không hỏi về từng phần tử. Phần tử nhỏ hơn `left` **được phép** có mặt, miễn max vẫn trong đoạn.

In [27]:
nums, left, right = [2, 1, 4, 3], 2, 3
print("sai  (mọi phần tử trong đoạn):", wrong_all_in_range(nums, left, right))
print("đúng                         :", Solution().numSubarrayBoundedMax(nums, left, right))

sai  (mọi phần tử trong đoạn): 2
đúng                         : 3


**Vì sao sai.** `[2, 1]` có max `2 ∈ [2, 3]` nên hợp lệ, dù `1 < left`. **Cách sửa:** chỉ coi phần tử **`> right`** là bức tường; phần tử nhỏ được xử lý bằng `last_ok` (hoặc bù trừ).

### Lỗi 2: coi phần tử nhỏ hơn `left` là bức tường

Cũng là hiểu sai tương tự: reset đếm khi `x < left` làm mất các mảng con `[2, 1]`, `[2, 1, 2]`,...

In [28]:
nums, left, right = [3, 1, 2, 1], 1, 2
print("sai  :", wrong_small_is_wall([2, 1, 4, 3], 2, 3), "(đúng là 3)")
print("đúng :", Solution().numSubarrayBoundedMax([2, 1, 4, 3], 2, 3))
print("ví dụ 4 (nums = [3,1,2,1], [1, 2]) ->", Solution().numSubarrayBoundedMax(nums, left, right))

sai  : 2 (đúng là 3)
đúng : 3
ví dụ 4 (nums = [3,1,2,1], [1, 2]) -> 6


**Cách sửa.** Dùng `last_ok` để nhớ phần tử vừa gần nhất: một phần tử nhỏ **không** reset gì cả.

### Lỗi 3: bù trừ nhưng quên `- 1` ở `left`

`f(right) - f(left)` trừ luôn cả các mảng con có max **đúng bằng `left`** — mà những mảng con này hợp lệ và phải giữ lại. Phải trừ `f(left - 1)` (max **nhỏ hơn hẳn** `left`).

In [29]:
nums, left, right = [2, 1, 4, 3], 2, 3
print("sai  (f(right) - f(left)):", wrong_complement(nums, left, right))
print("đúng (f(right) - f(left - 1)):", Solution2().numSubarrayBoundedMax(nums, left, right))

sai  (f(right) - f(left)): 1
đúng (f(right) - f(left - 1)): 3


**Vì sao sai.** Với `left = 2`: `f(2)` đếm cả `[2]`, `[2, 1]` (max = 2), nên bị trừ oan khỏi đáp án. **Cách sửa:** `f(left - 1)`. (Nhờ vậy khi `left = 0` ta có `f(-1) = 0`.)

### Lỗi 4: đếm cả mảng con **không có** phần tử vừa

Chỉ biết `last_big` rồi cộng `i - last_big` sẽ đếm cả các mảng con gồm toàn phần tử nhỏ `< left`.

In [30]:
print("sai  :", wrong_no_last_ok([1], 2, 3), "(phải là 0: [1] có max 1 < left)")
print("đúng :", Solution().numSubarrayBoundedMax([1], 2, 3))

sai  : 1 (phải là 0: [1] có max 1 < left)
đúng : 0


**Vì sao sai.** `[1]` chưa có phần tử nào `≥ left`, nên max `1 < 2`. **Cách sửa:** điều kiện **phải có** một phần tử vừa trong mảng con, tức `s ≤ last_ok`; cộng `last_ok - last_big` (chứ không phải `i - last_big`).

### Lỗi 5: ngăn xếp đơn điệu đếm **trùng** khi có phần tử bằng nhau

Nếu ở cả hai phía đều tìm phần tử **lớn hơn hẳn** (cùng quy tắc `>`) thì mảng con có nhiều phần tử max bằng nhau được gán cho **nhiều** phần tử. Phải chọn một phía "chặt" (`>`), phía còn lại "lỏng" (`≥`).

In [31]:
nums, left, right = [2, 2, 2], 2, 2
print("sai  :", wrong_stack_dup(nums, left, right))
print("đúng :", Solution4().numSubarrayBoundedMax(nums, left, right), "(= 3 * 4 / 2 = 6)")

sai  : 10
đúng : 6 (= 3 * 4 / 2 = 6)


**Vì sao sai.** Với `[2, 2, 2]` mỗi phần tử coi mình là max của mọi mảng con chứa nó và chạm tới biên, nên tổng vượt quá `6`. **Cách sửa:** `prev` dùng `≥` (dừng ở phần tử bằng bên trái) và `nxt` dùng `>` (dừng ở phần tử lớn hơn hẳn bên phải) — như `Solution4`.

### Lỗi 6: `O(n²)` với `n = 10^5`

Thử mọi mảng con (cách 1) quá chậm khi các phần tử đều `≤ right`: `~5 × 10^9` phép. Hãy ước lượng độ phức tạp **trước khi** code.

In [32]:
for n in (500, 1000, 2000, 4000):
    nums = [1] * n
    t0 = time.perf_counter()
    count_brute(nums, 1, 1)
    t1 = time.perf_counter()
    Solution().numSubarrayBoundedMax(nums, 1, 1)
    t2 = time.perf_counter()
    print(f"n = {n:>5}: brute {(t1 - t0) * 1000:8.1f} ms | O(n) {(t2 - t1) * 1000:6.3f} ms")

n =   500: brute     15.6 ms | O(n)  0.060 ms
n =  1000: brute     62.4 ms | O(n)  0.125 ms


n =  2000: brute    168.0 ms | O(n)  0.163 ms


n =  4000: brute    598.1 ms | O(n)  0.534 ms


**Vì sao chậm.** Khi nhân đôi `n`, thời gian brute tăng gần gấp 4 (bậc hai), còn `O(n)` tăng gấp 2. Với `n = 10^5` brute cần cỡ `10^5 × 10^5 / 2` phép — vài chục phút. **Cách sửa:** dùng một trong bốn cách `O(n)`.

### Lỗi 7: không xử lý `left = 0`

Với bù trừ, `f(left - 1) = f(-1)` phải bằng `0`. Nếu bạn viết `f` cho điều kiện kiểu `x <= bound` thì `f(-1) = 0` tự nhiên (vì `nums[i] ≥ 0`). Nhưng nếu `f` dùng giá trị khởi tạo `max_so_far = 0` hay so sánh khác đi thì `left = 0` có thể sai.

In [33]:
nums = [0, 0, 1]
print(Solution2().numSubarrayBoundedMax(nums, 0, 0), "(chỉ [0], [0], [0,0] -> 3)")
assert Solution2().numSubarrayBoundedMax(nums, 0, 0) == oracle_cubic(nums, 0, 0) == 3
assert Solution().numSubarrayBoundedMax(nums, 0, 1) == oracle_cubic(nums, 0, 1) == 6

3 (chỉ [0], [0], [0,0] -> 3)


**Cách sửa.** Luôn thử `left = 0` và `left == right` trong test.

## 14. Mẹo & nhận dạng pattern

- **"Đếm mảng con có max/min trong khoảng"** → **bù trừ** `f(right) - f(left - 1)` với `f(b)` = đếm mảng con có max `≤ b` (đoạn liên tiếp các phần tử `≤ b`, cộng `run`).
- **Đếm theo điểm kết thúc**: duyệt `i`, hỏi "có bao nhiêu điểm bắt đầu hợp lệ?" — thường là hiệu hai "vị trí gần nhất" (`last_ok - last_big`).
- **Phân loại phần tử thành ba loại** (nhỏ/vừa/lớn) so với đoạn `[left, right]` rồi suy ra điều kiện mảng con: "không có lớn" + "có ít nhất một vừa".
- **Đóng góp của từng phần tử làm "max"/"min"** bằng ngăn xếp đơn điệu: `(i - prev) × (nxt - i)`; nhớ quy tắc **một phía `≥`, một phía `>`** để không đếm trùng khi có giá trị bằng nhau.
- **Đáp án có thể lớn (tổng bậc hai)**: Python không tràn; ở C++/Java dùng `long`.
- **Kiểm chứng**: viết oracle `O(n³)` bằng `max(slice)` cho `n ≤ 14` và đối chiếu.
- Họ hàng: **907 Sum of Subarray Minimums**, **2104 Sum of Subarray Ranges**, **713 Subarray Product Less Than K**, **992 Subarrays with K Different Integers** (bù trừ "tối đa K" trừ "tối đa K−1"), **1248 Count Number of Nice Subarrays**.

## 15. Biến thể

### 15.1. Số mảng con có max **đúng bằng** `k`

Chính là bù trừ với `left = right = k`.

In [34]:
def count_max_equal(nums, k):
    return Solution().numSubarrayBoundedMax(nums, k, k)

print(count_max_equal([2, 1, 4, 3], 3), count_max_equal([2, 1, 4, 3], 4))      # 1  6
assert count_max_equal([2, 1, 4, 3], 3) == 1 == oracle_cubic([2, 1, 4, 3], 3, 3)
assert count_max_equal([2, 1, 4, 3], 4) == 6 == oracle_cubic([2, 1, 4, 3], 4, 4)

1 6


### 15.2. Độ dài **lớn nhất** của một mảng con có max trong `[left, right]`

Mảng con hợp lệ dài nhất là một **đoạn tối đa không có phần tử lớn** (nếu nó chứa ít nhất một phần tử vừa).

In [35]:
def longest_bounded(nums, left, right):
    best = start = 0
    has_ok = False
    for i, x in enumerate(nums + [float("inf")]):         # phần tử lính canh vô cực kết thúc đoạn cuối
        if x > right:
            if has_ok:
                best = max(best, i - start)
            start, has_ok = i + 1, False
        elif x >= left:
            has_ok = True
    return best

print(longest_bounded([2, 9, 2, 5, 6], 2, 8))              # 3 ([2,5,6])
assert longest_bounded([2, 9, 2, 5, 6], 2, 8) == 3
assert longest_bounded([1, 1, 1], 2, 3) == 0
rng = random.Random(152)
for _ in range(300):
    nums, l, r = random_case(rng, rng.randint(1, 10), rng.choice([2, 5]))
    brute = max((j - i + 1 for i in range(len(nums)) for j in range(i, len(nums)) if l <= max(nums[i:j + 1]) <= r), default=0)
    assert longest_bounded(nums, l, r) == brute, (nums, l, r)
print("longest_bounded khớp brute")

3
longest_bounded khớp brute


### 15.3. Số mảng con có **min** trong `[left, right]`

Đối xứng: đổi vai trò (phần tử bé hơn `left` là "tường"). Nghĩa là trên mảng `-nums` bài này thành bài cũ với đoạn `[-right, -left]`.

In [36]:
def count_min_in_range(nums, left, right):
    return Solution().numSubarrayBoundedMax([-x for x in nums], -right, -left)

def oracle_min(nums, left, right):
    n = len(nums)
    return sum(1 for i in range(n) for j in range(i, n) if left <= min(nums[i:j + 1]) <= right)

# lưu ý: số âm nằm ngoài ràng buộc của bài gốc nhưng thuật toán vẫn đúng cho số nguyên bất kỳ
# (chỉ cần last_big/last_ok khởi tạo -1 và so sánh trực tiếp)
rng = random.Random(153)
for _ in range(300):
    nums = [rng.randint(0, 6) for _ in range(rng.randint(1, 10))]
    l = rng.randint(0, 6); r = rng.randint(l, 6)
    assert count_min_in_range(nums, l, r) == oracle_min(nums, l, r), (nums, l, r)
print("min trong khoảng: OK")

min trong khoảng: OK


### 15.4. Đếm mảng con có max `≤ k` ít nhất một lần chạm `k`

Tức max đúng bằng `k` (đã làm ở 15.1). Mở rộng: **tích** các phần tử `< k` (bài 713) dùng hai con trỏ — cùng khung "đoạn liên tiếp thỏa tính chất kế thừa".

### 15.5. Biến thể để tự nghĩ

- **Hai mảng**: mảng con chung có max trong đoạn?
- **Cửa sổ có độ dài cố định `k`**: số cửa sổ có max trong `[left, right]` (dùng deque đơn điệu).
- **Trả về chính các mảng con** (tọa độ `(s, e)`) — chỉ khả thi khi số lượng nhỏ.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi mảng con | max chạy, dừng sớm khi `> right` | O(n²) | O(1) | oracle/hiểu đề, `n` nhỏ |
| 2. Bù trừ | `f(right) - f(left - 1)` với `f` đếm theo `run` | O(n) | O(1) | khi muốn code đơn giản, ít lỗi chỉ số |
| 3. Hai chỉ số `last_big`, `last_ok` | điểm bắt đầu hợp lệ `(last_big, last_ok]` | O(n) | O(1) | **bản nộp** |
| 4. Ngăn xếp đơn điệu | đóng góp `(i - prev) × (nxt - i)` của mỗi max | O(n) | O(n) | khung tổng quát cho "tổng theo max/min" |
| 5. DP theo điểm kết thúc | `dp[i]` theo ba loại phần tử | O(n) | O(1) | khi muốn dạng truy hồi để chứng minh |

**Lời khuyên.** Nhớ cách 2 (bù trừ, `left - 1`) và cách 3 (hai chỉ số); biết cách 4 vì nó dùng được cho cả họ bài "tổng/đếm theo max-min" (907, 2104).

## 17. 5 bài tập vận dụng (dễ → khó)

Cell **TODO** chỉ gán `None` hoặc định nghĩa hàm rỗng, **không được gọi** khi chạy notebook. Đáp án ở mục 19.

**Bài 1 (đoán output).** Dự đoán rồi chạy:

In [37]:
print(Solution().numSubarrayBoundedMax([2, 1, 4, 3], 2, 3))
print(Solution().numSubarrayBoundedMax([5, 5, 5], 5, 5))
print(Solution().numSubarrayBoundedMax([1, 2, 3], 4, 9))

3
6
0


**Bài 2 (tính tay).** Với `nums = [3, 1, 2, 1]`, `left = 1`, `right = 2` (ví dụ 4), hãy tính tay `f(right)` và `f(left - 1)` (số mảng con có mọi phần tử `≤ 2` và `≤ 0`) rồi gán **đáp án** vào `ans2` (đang là `None`).

In [38]:
ans2 = None      # TODO: một số nguyên

**Bài 3 (điền TODO).** Viết `count_at_most(nums, bound)` trả về **số mảng con có mọi phần tử `≤ bound`** (tức có max `≤ bound`). Ví dụ `count_at_most([2, 9, 2, 5, 6], 8) == 7`.

In [39]:
def count_at_most(nums, bound):
    # TODO: giữ run = độ dài đoạn liên tiếp <= bound kết thúc tại vị trí hiện tại; cộng run vào tổng
    pass

**Bài 4 (tự viết lại).** Không nhìn mục 7, tự viết `my_count(nums, left, right)` theo cách **hai chỉ số** `last_big`, `last_ok` (cách 3).

In [40]:
def my_count(nums, left, right):
    # TODO
    pass

**Bài 5 (mini challenge).** (a) `count_window_max_in_range(nums, k, left, right)`: đếm số **cửa sổ độ dài đúng `k`** (mảng con liên tiếp gồm `k` phần tử) có max thuộc `[left, right]`. (b) `count_subarrays_len_at_least_2(nums, left, right)`: đếm số mảng con có max trong `[left, right]` **và độ dài `≥ 2`**.

In [41]:
def count_window_max_in_range(nums, k, left, right):
    # TODO
    pass

def count_subarrays_len_at_least_2(nums, left, right):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: ví dụ thứ hai toàn `5` và `left = right = 5`: mọi mảng con thỏa.
- Bài 2: `f(2)` đếm theo đoạn liên tiếp các số `≤ 2`: đoạn `[1, 2, 1]`.
- Bài 3: `run = run + 1 if x <= bound else 0`.
- Bài 4: hai biến `last_big`, `last_ok`, khởi tạo `-1`.
- Bài 5: (a) tính max cửa sổ bằng `max(nums[i:i+k])`; (b) lấy đáp án gốc trừ đi số mảng con độ dài `1` hợp lệ.

**Mức 2 (vừa):**
- Bài 2: `f(2)` = đoạn `[1,2,1]` có `3 · 4 / 2 = 6` mảng con (3 bị chặn bởi 3 ở đầu). `f(0)` = 0 (không phần tử nào `≤ 0`). Đáp án `6 - 0 = 6`.
- Bài 3: `total += run` sau mỗi phần tử.
- Bài 4: `if x > right: last_big = i elif x >= left: last_ok = i`; rồi `if last_ok > last_big: ans += last_ok - last_big`.
- Bài 5: (a) `sum(1 for i in range(len(nums) - k + 1) if left <= max(nums[i:i + k]) <= right)` là bản chậm; bản nhanh dùng deque; (b) số mảng con độ dài 1 hợp lệ là `sum(1 for x in nums if left <= x <= right)`.

**Mức 3 (gần đáp án):**
- Bài 3: xem `count` trong cách 2.
- Bài 4: xem mục 9, bỏ chú thích.
- Bài 5: (b) `Solution().numSubarrayBoundedMax(nums, left, right) - sum(left <= x <= right for x in nums)`.

## 19. Đáp án tham khảo

Chạy kiểm tra bằng `assert`.

In [42]:
# Bài 1
assert Solution().numSubarrayBoundedMax([2, 1, 4, 3], 2, 3) == 3
assert Solution().numSubarrayBoundedMax([5, 5, 5], 5, 5) == 6
assert Solution().numSubarrayBoundedMax([1, 2, 3], 4, 9) == 0

# Bài 2: f(2) = 6 (đoạn [1,2,1]), f(0) = 0 -> 6
ans2 = 6
assert ans2 == Solution().numSubarrayBoundedMax([3, 1, 2, 1], 1, 2) == oracle_cubic([3, 1, 2, 1], 1, 2)
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [43]:
# Bài 3
def count_at_most(nums, bound):
    total = run = 0
    for x in nums:
        run = run + 1 if x <= bound else 0
        total += run
    return total

assert count_at_most([2, 9, 2, 5, 6], 8) == 7
assert count_at_most([3, 1, 2, 1], 2) == 6
assert count_at_most([3, 1, 2, 1], 0) == 0
assert count_at_most([], 5) == 0
rng = random.Random(3)
for _ in range(200):
    nums, l, r = random_case(rng, rng.randint(1, 10), 5)
    assert count_at_most(nums, r) - count_at_most(nums, l - 1) == oracle_cubic(nums, l, r)
print("Bài 3 OK")

Bài 3 OK


In [44]:
# Bài 4
def my_count(nums, left, right):
    ans, last_big, last_ok = 0, -1, -1
    for i, x in enumerate(nums):
        if x > right:
            last_big = i
        elif x >= left:
            last_ok = i
        if last_ok > last_big:
            ans += last_ok - last_big
    return ans

rng = random.Random(4)
for _ in range(300):
    nums, l, r = random_case(rng, rng.randint(1, 12), rng.choice([2, 5, 10**9]))
    assert my_count(nums, l, r) == oracle_cubic(nums, l, r)
print("Bài 4 OK")

Bài 4 OK


In [45]:
# Bài 5 (a): bản đơn giản O(n*k); bản nhanh dùng deque đơn điệu
from collections import deque

def count_window_max_in_range(nums, k, left, right):
    dq, count = deque(), 0                         # dq: chỉ số, giá trị giảm dần
    for i, x in enumerate(nums):
        while dq and nums[dq[-1]] <= x:
            dq.pop()
        dq.append(i)
        if dq[0] <= i - k:
            dq.popleft()
        if i >= k - 1 and left <= nums[dq[0]] <= right:
            count += 1
    return count

def windows_oracle(nums, k, left, right):
    return sum(1 for i in range(len(nums) - k + 1) if left <= max(nums[i:i + k]) <= right)

assert count_window_max_in_range([2, 1, 4, 3], 2, 2, 3) == windows_oracle([2, 1, 4, 3], 2, 2, 3) == 1
assert count_window_max_in_range([2, 1, 4, 3], 1, 2, 3) == 2
rng = random.Random(51)
for _ in range(300):
    nums, l, r = random_case(rng, rng.randint(1, 12), rng.choice([2, 5, 10]))
    k = rng.randint(1, len(nums))
    assert count_window_max_in_range(nums, k, l, r) == windows_oracle(nums, k, l, r)

# Bài 5 (b)
def count_subarrays_len_at_least_2(nums, left, right):
    singles = sum(1 for x in nums if left <= x <= right)
    return Solution().numSubarrayBoundedMax(nums, left, right) - singles

assert count_subarrays_len_at_least_2([2, 1, 4, 3], 2, 3) == 1            # chỉ [2, 1]
assert count_subarrays_len_at_least_2([5, 5, 5], 5, 5) == 3               # [5,5], [5,5], [5,5,5]
for _ in range(300):
    nums, l, r = random_case(rng, rng.randint(1, 10), rng.choice([2, 5]))
    brute = sum(1 for i in range(len(nums)) for j in range(i + 1, len(nums)) if l <= max(nums[i:j + 1]) <= r)
    assert count_subarrays_len_at_least_2(nums, l, r) == brute
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (hai chỉ số, một lượt):**

```python
class Solution:
    def numSubarrayBoundedMax(self, nums: List[int], left: int, right: int) -> int:
        ans = 0
        last_big = last_ok = -1          # vị trí gần nhất của phần tử > right / thuộc [left, right]
        for i, x in enumerate(nums):
            if x > right:
                last_big = i
            elif x >= left:
                last_ok = i
            if last_ok > last_big:
                ans += last_ok - last_big
        return ans
```

**Template (bù trừ):**

```python
def count(bound):
    total = run = 0
    for x in nums:
        run = run + 1 if x <= bound else 0
        total += run
    return total

return count(right) - count(left - 1)
```

**Câu thần chú:** *không chứa phần tử `> right`, và chứa ít nhất một phần tử trong `[left, right]`; đếm theo điểm kết thúc `i`: `last_ok - last_big`; hoặc `f(right) - f(left - 1)`.*

**Checklist trước khi submit:**

- [ ] Chỉ phần tử `> right` là "bức tường"; phần tử `< left` **không** reset gì?
- [ ] Số điểm bắt đầu hợp lệ là `last_ok - last_big` chỉ khi `last_ok > last_big`?
- [ ] Nếu bù trừ: trừ `f(left - 1)` (không phải `f(left)`)?
- [ ] Nếu dùng ngăn xếp: một phía `≥`, một phía `>` để không đếm trùng?
- [ ] Độ phức tạp `O(n)` (không dùng `O(n²)` với `n = 10^5`)?
- [ ] Đã thử: `n = 1`, `left = 0`, `left == right`, toàn phần tử bằng nhau, toàn phần tử lớn/nhỏ, ví dụ của đề?